<a id="top"></a>

# Lab L2306: Compose a skill system and audit it

```yaml
title: "Lab L2306: Compose a skill system and audit it"
keywords: skill composition, sequential handoff, shared sub-skill, dependency graph, just-in-time loading, over-chaining, description collision, skill-vs-tool, capstone, agent skills
estimated duration: 40
```

> **Lesson:** L23 — Skill patterns & composition (composition capstone).
> **Roadmap:** [objectives.md](../../../../docs/origin/lesson_roadmaps/L23/objectives.md).
> **Anchor model: Claude Sonnet 4.6.**
>
> **Runs two ways.** By default this notebook drives the offline scripted `FakeModel` (no key, deterministic, restart-and-run-all passes). Set `ANTHROPIC_API_KEY` and the optional live cells drive real Sonnet 4.6.

**You will be able to:**

- **Compose** skills two ways: a **sequential handoff** (A → B) and a **shared lower-level skill** (A → C ← B).
- **Draw the dependency graph** of a small skill system and name its pipelines, shared (fan-in) nodes, and leaves.
- Reason about **just-in-time loading across the graph** in concrete numbers (always-on cost vs. bodies loaded on the path).
- **Self-audit** a skill system for the three composition anti-patterns: over-chaining, a shared "skill" that's really a tool, and description collisions.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 - A sequential handoff](#2-problem-1---a-sequential-handoff)
- [3. Problem 2 - A shared lower-level skill](#3-problem-2---a-shared-lower-level-skill)
- [4. Problem 3 - Draw the dependency graph](#4-problem-3---draw-the-dependency-graph)
- [5. Problem 4 - JIT in numbers](#5-problem-4---jit-in-numbers)
- [6. Problem 5 - Self-audit for the three anti-patterns](#6-problem-5---self-audit-for-the-three-anti-patterns)
- [7. Self-check](#7-self-check)

## 1. Setup

This is the **composition capstone**. You start with a tiny, runnable skill *system* — a stand-in for this repo's real curriculum-authoring skills — and grow it into a small system of your own, then graph and audit it.

The runtime is the same one from the L2304 and L2305 lectures, so you already know it:

- **`Skill`** — a frozen dataclass: `name`, `description` (says *when* it applies — the trigger, from [L22](../L22/L2201_intro.md)), and `body` (the instructions, loaded on demand).
- **`REGISTRY`** — the starter skill system. Two **operating skills** (`author-lesson-roadmap`, `generate-materials-from-roadmap`) plus one small leaf, `check-lesson-links`. An edge "skill A invokes skill B" is literally *A's body calling `load_skill("B")`.*
- **`make_skill_tools(registry)`** — builds the two LangChain tools the agent uses: `list_skills()` (discovery — returns every `name: description`) and `load_skill(name)` (load — reads one body into context). These two schemas are the **only always-on skill cost**.
- **`loaded_skills(messages)`** — reads a finished run's history and returns the skills the agent actually pulled, **in order**.
- **`dependency_edges(registry)`** — scans every body for `load_skill("…")` and returns the `(A, B)` edges — the map of the system.
- **`FakeModel` helpers** (`text_reply`, `tool_call`, `tool_reply`) — drive the agent **offline**, no key, by scripting exactly which tool calls it makes. This is how every problem below runs deterministically.

Run this cell as-is; do not edit it. Everything the problems need is defined here.

In [ ]:
from __future__ import annotations

import re
from collections.abc import Callable
from dataclasses import dataclass
from typing import Annotated

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage

from fluffy_potato_curriculum.common.config import get_settings
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)

SONNET = "claude-sonnet-4-6"
LIVE = bool(get_settings().anthropic_api_key)  # optional live path; offline is default


@dataclass(frozen=True)
class Skill:
    """A skill: a name, a description that says WHEN it applies, and a body."""

    name: str
    description: str
    body: str


# The STARTER skill system. A tiny stand-in for this repo's real curriculum-
# authoring skills. Bodies reference load_skill("X") so the dependency edges are
# literal, runnable text. Note: `generate-materials-from-roadmap` does NOT yet
# call the checker -- you build that shared fan-in yourself in Problem 2.
REGISTRY: dict[str, Skill] = {
    "author-lesson-roadmap": Skill(
        name="author-lesson-roadmap",
        description="Draft a lesson's roadmap docs (objectives, demos) from the PRD. Stage 1.",
        body=(
            "1. Read the PRD row and format specs.\n"
            "2. Draft objectives.md and demos_or_activities.md.\n"
            "3. Validate cross-references: load_skill('check-lesson-links').\n"
            "4. Hand off to stage 2: load_skill('generate-materials-from-roadmap')."
        ),
    ),
    "generate-materials-from-roadmap": Skill(
        name="generate-materials-from-roadmap",
        description="Generate student materials (lectures, labs) from a roadmap. Stage 2.",
        body=(
            "1. Read the roadmap docs produced by stage 1.\n"
            "2. Generate intro, lectures, and lab pairs."
        ),
    ),
    "check-lesson-links": Skill(
        name="check-lesson-links",
        description="Check one document's markdown links; judge each unresolved one. Shared.",
        body="Run extract_links.py, then classify each unresolved link (broken / placeholder).",
    ),
}


def make_skill_tools(registry: dict[str, Skill]) -> list[Callable[..., str]]:
    """Build the two JIT tools over a skill registry: discovery + load."""

    def list_skills() -> str:
        """Discovery: list every registered skill as `name: description`.

        Call this first to see what is available before choosing one to load.
        """
        return "\n".join(f"{s.name}: {s.description}" for s in registry.values())

    def load_skill(
        name: Annotated[str, "the exact skill name from list_skills to read into context"],
    ) -> str:
        """Load: read one skill's full body into context, or an error if unknown."""
        skill = registry.get(name)
        if skill is None:
            return f"no such skill: {name!r}"
        return skill.body

    return [list_skills, load_skill]


def loaded_skills(messages: list[AIMessage]) -> list[str]:
    """Which skills the agent actually pulled, in order, from the run history."""
    return [
        call["args"]["name"]
        for msg in messages
        if isinstance(msg, AIMessage)
        for call in msg.tool_calls
        if call["name"] == "load_skill"
    ]


def dependency_edges(registry: dict[str, Skill]) -> list[tuple[str, str]]:
    """Extract 'A -> B' edges: skill A's body that calls load_skill('B')."""
    pattern = re.compile(r"load_skill\(['\"]([^'\"]+)['\"]\)")
    edges: list[tuple[str, str]] = []
    for skill in registry.values():
        for target in pattern.findall(skill.body):
            edges.append((skill.name, target))
    return edges


SYSTEM = (
    "You are the curriculum-build agent. Call list_skills() to discover skills, "
    "then load_skill(name) to read the one you need and follow it."
)

print("Starter registry:", list(REGISTRY))
print("Starter edges:", dependency_edges(REGISTRY))
print("LIVE:", LIVE)

[↑ Back to top](#top)

## 2. Problem 1 - A sequential handoff

A **sequential handoff** is A → B: one skill produces something a second skill consumes; the *pipeline* is the capability (like this repo's real `author-lesson-roadmap` → `generate-materials-from-roadmap` two-stage flow). At runtime the edge is literal — A's body tells the agent to `load_skill("B")`.

Grow the system: author a **new operating skill** and hand it off to an existing one.

1. Define a new `Skill` — call it `review-lesson` — whose body ends by handing off to `generate-materials-from-roadmap` via `load_skill('generate-materials-from-roadmap')`. (Fixes found in review get regenerated by stage 2.)
2. Add it to `REGISTRY`.
3. Script a `FakeModel` run that walks the handoff: `list_skills`, then `load_skill` on your new skill, then `load_skill` on its target, then a `text_reply` final answer.
4. Build the agent with `create_agent` and run it; print `loaded_skills(messages)`.

The `assert` checks the loaded path is exactly your skill followed by its handoff target.

In [ ]:
# 1-2. Author the new operating skill and register it. Its body must HAND OFF to
# generate-materials-from-roadmap via a literal load_skill(...) call in its text.
# TODO: replace the body with a short numbered procedure whose last step is
#       "load_skill('generate-materials-from-roadmap')".
REGISTRY["review-lesson"] = Skill(
    name="review-lesson",
    description="Review a finished lesson end to end, then hand off fixes to stage 2. Stage 3.",
    body="",  # TODO: your handoff body here
)
raise NotImplementedError("your code here")

# 3. Script a run that walks the handoff: discover -> load mine -> load its target -> answer.
# TODO: build a FakeModel([...]) with, in order:
#   - tool_reply(tool_call("c1", "list_skills", {}))
#   - tool_reply(tool_call("c2", "load_skill", {"name": "review-lesson"}))
#   - tool_reply(tool_call("c3", "load_skill", {"name": "generate-materials-from-roadmap"}))
#   - text_reply("...done.")
p1_model = ...  # TODO

# 4. Build the agent, run it, and read back the loaded path.
p1_agent = create_agent(p1_model, make_skill_tools(REGISTRY), system_prompt=SYSTEM)
p1_messages = (await p1_agent.ainvoke({"messages": [HumanMessage(content="Review lesson L99.")]}))[
    "messages"
]
p1_path = loaded_skills(p1_messages)
print("loaded path:", p1_path)

assert p1_path == ["review-lesson", "generate-materials-from-roadmap"], (
    "the handoff path must load your new skill, then its target, in that order"
)
print("Problem 1 OK: sequential handoff review-lesson -> generate-materials-from-roadmap")

[↑ Back to top](#top)

## 3. Problem 2 - A shared lower-level skill

The other composition move is a **shared lower-level skill**: one small skill that *several* operating skills invoke — write-once, reuse-many. That is **A → C ← B**: two callers, one dependency (a *fan-in* node), not a pipeline.

Right now `check-lesson-links` (`C`) is called by only one operating skill — `author-lesson-roadmap`. Make it genuinely shared by wiring a **second caller**:

1. Edit `generate-materials-from-roadmap`'s body so it *also* ends by validating cross-references with `load_skill('check-lesson-links')`. (`Skill` is frozen, so replace the whole entry in `REGISTRY`.)
2. Script a run whose path uses `C` **twice** — once after `author-lesson-roadmap`, once after `generate-materials-from-roadmap`.
3. Run it and print the loaded path.

The `assert` checks `C` was loaded twice — a shared skill loads **once per use**, not once per caller.

In [ ]:
# 1. Wire a SECOND caller into the shared node: edit generate's body so it too
#    ends with load_skill('check-lesson-links'). `Skill` is frozen, so REPLACE
#    the whole REGISTRY entry (don't mutate the existing one).
# TODO: set the body to a numbered procedure whose last step is
#       "load_skill('check-lesson-links')".
REGISTRY["generate-materials-from-roadmap"] = Skill(
    name="generate-materials-from-roadmap",
    description="Generate student materials (lectures, labs) from a roadmap. Stage 2.",
    body="",  # TODO: your body, ending in load_skill('check-lesson-links')
)
raise NotImplementedError("your code here")

# 2. Script a path that USES the shared skill twice (once per operating skill):
#    list_skills -> load author -> load check -> load generate -> load check -> answer.
# TODO: build the FakeModel([...]) for that path.
p2_model = ...  # TODO

# 3. Run it and read the loaded path.
p2_agent = create_agent(p2_model, make_skill_tools(REGISTRY), system_prompt=SYSTEM)
p2_messages = (
    await p2_agent.ainvoke({"messages": [HumanMessage(content="Build and check L99.")]})
)["messages"]
p2_path = loaded_skills(p2_messages)
print("loaded path:", p2_path)

assert p2_path.count("check-lesson-links") == 2, (
    "a shared skill loads once PER USE, not once per caller -- expected two loads of C"
)
print("Problem 2 OK: check-lesson-links is now a shared fan-in node (A -> C <- B)")

[↑ Back to top](#top)

## 4. Problem 3 - Draw the dependency graph

After Problems 1-2 your `REGISTRY` is a real little system: `author-lesson-roadmap`, `generate-materials-from-roadmap`, `review-lesson`, and the shared `check-lesson-links`. Now *map* it. Call `dependency_edges(REGISTRY)` and sort the edges into three buckets:

- **`shared_nodes`** — fan-in targets: any skill pointed at by **more than one** distinct caller (the DRY reuse points). This is the fan-in you built in Problem 2.
- **`pipelines`** — handoff edges between two *operating* skills (the target is not the leaf `check-lesson-links`).
- **`leaves`** — skills whose own body invokes nothing (they only get called).

The `assert` checks the shared node you built in Problem 2 shows up in `shared_nodes`.

In [ ]:
edges = dependency_edges(REGISTRY)
print("edges:", edges)

# TODO: fill in the three buckets from `edges`.
#   - shared_nodes: sorted list of targets with MORE THAN ONE distinct caller (fan-in).
#   - pipelines:    sorted list of (A, B) edges where B != "check-lesson-links".
#   - leaves:       sorted list of skill names that never appear as a caller in `edges`.
shared_nodes: list[str] = ...  # TODO
pipelines: list[tuple[str, str]] = ...  # TODO
leaves: list[str] = ...  # TODO
raise NotImplementedError("your code here")

print("shared_nodes:", shared_nodes)
print("pipelines:   ", pipelines)
print("leaves:      ", leaves)

assert "check-lesson-links" in shared_nodes, (
    "the fan-in you wired in Problem 2 must appear as a shared node"
)
print("Problem 3 OK: graph mapped; check-lesson-links is the shared fan-in node")

[↑ Back to top](#top)

## 5. Problem 4 - JIT in numbers

This is the cost payoff, made concrete. Two facts from the L2304 lecture:

- **Always-on cost is just the two tool schemas** (`list_skills`, `load_skill`) — *not* the skills. It does not grow as you add skills; the registry's `{name, description}` list is paid only when the agent calls `list_skills`, and each body is paid only when `load_skill` actually pulls it.
- Along a given path, the **body footprint** is the sum of the **distinct** bodies loaded. A shared skill that loads twice (Problem 2) is the **same body** both times, so it counts **once** toward the footprint.

Compute, for your Problem 2 path (`p2_path`):

1. `always_on` — a one-line string naming what stays always-on.
2. `body_chars` — total characters of the **unique** bodies loaded along `p2_path`.

The `assert` checks your `body_chars` ignores the repeat load of the shared skill.

In [ ]:
# 1. Always-on cost does NOT include the skills -- only the two tool schemas.
# TODO: set always_on to a one-line string naming what stays always-on.
always_on = ...  # TODO

# 2. Body footprint = sum of DISTINCT bodies loaded along p2_path (the shared skill
#    loads twice but is ONE body -> count it once). Hint: dict.fromkeys(p2_path)
#    dedupes while keeping order.
# TODO: compute p2_unique (the de-duplicated path) and body_chars (sum of len(body)).
p2_unique: list[str] = ...  # TODO
body_chars: int = ...  # TODO
raise NotImplementedError("your code here")

print("always-on:", always_on)
print("distinct bodies loaded:", p2_unique)
print("total body chars (unique):", body_chars)

assert body_chars == sum(len(REGISTRY[name].body) for name in set(p2_path)), (
    "the footprint must count each distinct body once -- not once per load"
)
print("Problem 4 OK: body footprint counts", len(p2_unique), "unique bodies")

[↑ Back to top](#top)

## 6. Problem 5 - Self-audit for the three anti-patterns

Composition is only a win if the system is **cheaper to run and easier to reason about** than the monolith it replaced. The three ways it silently gets worse (from the L2305 lecture):

- **Over-chaining** — so many tiny handoffs that the pipeline is mostly load-overhead and every hop is a place selection can fail. Cure: collapse hops; a handoff should exist because the seam is a *real reusable/testable boundary*.
- **A shared "skill" that's really a tool** — the shared node is a deterministic operation with structured I/O that every caller invokes the same way. That is a **tool** ([L07/L08](../L22/L2201_intro.md)), *called with a schema*, not *read as instructions*. Cure: make it a tool (or let its script be the interface) and drop the skill wrapper.
- **Description collisions** — two skills whose descriptions overlap so the model can't reliably pick; selection degrades across the *whole set*. Cure: make descriptions *mutually discriminating* — each says what it's for **and implicitly what it's not**.

**Part A (written).** Audit YOUR system (the `REGISTRY` you grew in Problems 1-3) against all three. For each: is it present, at risk, or clearly avoided — and why? Watch `check-lesson-links` in particular for the "really a tool" smell.

**Part B (code).** Below is a pair of **colliding descriptions**. Rewrite **one** of them so the two are mutually discriminating. The `assert` checks the two now share almost no content words.

_TODO: your audit here. For each anti-pattern — over-chaining, a shared "skill" that's really a tool, description collisions — say whether YOUR system is present / at-risk / avoided, and why. (Double-click to edit.)_

In [ ]:
# A colliding pair: both descriptions read as "draft the lesson's roadmap", so the
# model can't reliably tell them apart -- selection degrades across the whole set.
desc_a = "Draft the lesson's roadmap documents from the PRD."
desc_b = "Draft the lesson's roadmap and objectives documents."


def content_words(text: str) -> set[str]:
    """Lowercase word set, for a rough description-overlap measure."""
    return set(re.findall(r"[a-z]+", text.lower()))


overlap_before = len(content_words(desc_a) & content_words(desc_b))
print("shared content words before:", overlap_before)

# TODO: rewrite ONE description (assign desc_b_fixed) so the two are mutually
# discriminating -- say what it's for AND implicitly what it's NOT (a different
# input, output, and stage from desc_a), sharing almost no content words with it.
desc_b_fixed = ...  # TODO
raise NotImplementedError("your code here")

overlap_after = len(content_words(desc_a) & content_words(desc_b_fixed))
print("shared content words after:", overlap_after)

assert overlap_after < overlap_before, "the rewrite must reduce the shared-word overlap"
assert overlap_after <= 2, "the two descriptions should now share almost no content words"
print("Problem 5 OK: descriptions are now mutually discriminating")

[↑ Back to top](#top)

## 7. Self-check

There is **no autograder** here — the `assert`s in each problem are your first check (green means the mechanics are right), and the written audit in Problem 5 is judgment.

Compare your work against [`L2306_lab_solutions.ipynb`](L2306_lab_solutions.ipynb):

- **Problem 1** — your `review-lesson` body ends in `load_skill('generate-materials-from-roadmap')`, and the loaded path is exactly `["review-lesson", "generate-materials-from-roadmap"]`.
- **Problem 2** — `generate`'s body now ends in `load_skill('check-lesson-links')`, and the shared skill loads **twice** on your path.
- **Problem 3** — `check-lesson-links` is in `shared_nodes`; your pipelines and leaves match the graph you built.
- **Problem 4** — `always_on` names the two tool schemas; `body_chars` counts each distinct body once.
- **Problem 5** — your rewritten description shares almost no content words with `desc_a`, and your written audit names all three anti-patterns against your own system.

You have now composed a small skill **system** (a handoff *and* a shared sub-skill), mapped its dependency graph, priced its just-in-time loading, and audited it for the three anti-patterns — the L23 composition capstone.

[↑ Back to top](#top)